### Preprocessing Data CNBC

In [2]:
import re
import pandas as pd

KOMPAS_DATA_PATH = '../../data/data_berita/cleaning/kompas'

In [3]:
df = pd.read_csv(
    f"{KOMPAS_DATA_PATH}/kompas_labeled_manual_raw.csv",
    sep=",",
    encoding="latin1",
    engine="python"
)

df.head()


,id,date,title,text_preview,sentiment,confidence,notes,content,content_length
0,1,2022-01-23 00:00:00,Polemik Arteria Dahlan Jadi Pembelajaran Kader...,"BALI, KOMPAS.com- Sekretaris Jenderal PDI-P Ha...",netral,NaN,NaN,"BALI, KOMPAS.com- Sekretaris Jenderal PDI-P Ha...",618
1,2,2022-01-23 00:00:00,"HUT Ke-75 Megawati, Pramono Anung: Politik Ibu...","BALI, KOMPAS.com- Mantan Sekretaris Jenderal P...",netral,NaN,NaN,"BALI, KOMPAS.com- Mantan Sekretaris Jenderal P...",612
2,3,2022-01-24 00:00:00,Pengamat Tebak Pesan Anies ke Giring: Kalau Su...,"JAKARTA, KOMPAS.com- Direktur Eksekutif Parame...",netral,NaN,NaN,"JAKARTA, KOMPAS.com- Direktur Eksekutif Parame...",450
3,4,2022-01-24 00:00:00,UPDATE: 20.867 Kasus Aktif Covid-19 di Indonesia,"JAKARTA, KOMPAS.com- Pemerintah menyampaikan t...",negatif,NaN,NaN,"JAKARTA, KOMPAS.com- Pemerintah menyampaikan t...",652
4,5,2022-01-25 00:00:00,UPDATE 25 Januari: 7.483 Kasus Suspek Covid-19...,"JAKARTA, KOMPAS.com- Pemerintah mencatat, hing...",negatif,NaN,NaN,"JAKARTA, KOMPAS.com- Pemerintah mencatat, hing...",462


In [3]:
df.columns

Index(['id', 'date', 'title', 'text_preview', 'sentiment', 'confidence',
       'notes', 'content', 'content_length'],
      dtype='object')

In [4]:
#balikin id label data mich dlu ygy

#kita balikin id label datanya celo dlu ygy
df_raw = pd.read_csv('../../raw_fixed/kompas_articles.csv')

#lepas index
df_raw = df_raw.reset_index()
df_raw = df_raw.rename(columns={'index': 'id'})

df_raw = df_raw[['id', 'title']]
df_raw.columns = df_raw.columns.str.replace('title', 'raw_title')

#fetch df ama df_raw 
df['article_id'] = df['title'].apply(lambda x: df_raw.loc[df_raw['raw_title'] == x, 'id'].values[0] if x in df_raw['raw_title'].values else None)

In [5]:
df.head()

,id,date,title,text_preview,sentiment,confidence,notes,content,content_length,article_id
0,1,2022-01-23 00:00:00,Polemik Arteria Dahlan Jadi Pembelajaran Kader...,"BALI, KOMPAS.com- Sekretaris Jenderal PDI-P Ha...",netral,NaN,NaN,"BALI, KOMPAS.com- Sekretaris Jenderal PDI-P Ha...",618,9963.0
1,2,2022-01-23 00:00:00,"HUT Ke-75 Megawati, Pramono Anung: Politik Ibu...","BALI, KOMPAS.com- Mantan Sekretaris Jenderal P...",netral,NaN,NaN,"BALI, KOMPAS.com- Mantan Sekretaris Jenderal P...",612,9962.0
2,3,2022-01-24 00:00:00,Pengamat Tebak Pesan Anies ke Giring: Kalau Su...,"JAKARTA, KOMPAS.com- Direktur Eksekutif Parame...",netral,NaN,NaN,"JAKARTA, KOMPAS.com- Direktur Eksekutif Parame...",450,9956.0
3,4,2022-01-24 00:00:00,UPDATE: 20.867 Kasus Aktif Covid-19 di Indonesia,"JAKARTA, KOMPAS.com- Pemerintah menyampaikan t...",negatif,NaN,NaN,"JAKARTA, KOMPAS.com- Pemerintah menyampaikan t...",652,9945.0
4,5,2022-01-25 00:00:00,UPDATE 25 Januari: 7.483 Kasus Suspek Covid-19...,"JAKARTA, KOMPAS.com- Pemerintah mencatat, hing...",negatif,NaN,NaN,"JAKARTA, KOMPAS.com- Pemerintah mencatat, hing...",462,9936.0


In [6]:
# regex + clean function
URL_RE = re.compile(r"(https?://\S+|www\.\S+)", flags=re.IGNORECASE)
HTML_RE = re.compile(r"<[^>]+>")
WS_RE = re.compile(r"\s+")
SYMBOL_RE = re.compile(r"[^a-zA-Z0-9\s\.,!?\'\-]")

def clean_text(s: str) -> str:
    if pd.isna(s):
        return ""
    s = str(s)
    s = HTML_RE.sub(" ", s)
    s = URL_RE.sub(" ", s)
    s = SYMBOL_RE.sub(" ", s)
    s = s.lower()
    s = WS_RE.sub(" ", s).strip()
    return s

#cleaning title & content
df["title"] = df["title"].apply(clean_text)
df["content"] = df["content"].apply(clean_text)


# replace nama kolom
df.columns = df.columns.str.replace('sentiment', 'label')

#ambil cm date, title, content, label, article_id
df = df[['date', 'title', 'content', 'label', 'article_id']]

#bikin article id ke int
df['article_id'] = df['article_id'].fillna(0).astype('Int64')

# format article_id biar sama kek dapin jdi KOMPAS_XXXX
df["article_id"] = (
    df["article_id"]
    .astype(str)
    .str.replace(r"\.0$", "", regex=True)
)
df["article_id"] = pd.to_numeric(df["article_id"], errors="coerce").astype("Int64")

df["article_id"] = df["article_id"].apply(lambda x: f"KOMPAS_{int(x):04d}" if pd.notna(x) else pd.NA)

#fixing tanggal
df['date'] = pd.to_datetime(df['date']).dt.normalize()

#clean content data
df['content'] = df['content'].str.replace(r'^.*kompas\.com-', '', regex=True)



In [7]:
display(df.head())
display(df.tail())

,date,title,content,label,article_id
0,2022-01-23,polemik arteria dahlan jadi pembelajaran kader...,sekretaris jenderal pdi-p hasto kristiyanto m...,netral,KOMPAS_9963
1,2022-01-23,"hut ke-75 megawati, pramono anung politik ibu ...",mantan sekretaris jenderal pdi perjuangan pra...,netral,KOMPAS_9962
2,2022-01-24,pengamat tebak pesan anies ke giring kalau sum...,direktur eksekutif parameter politik indonesi...,netral,KOMPAS_9956
3,2022-01-24,update 20.867 kasus aktif covid-19 di indonesia,pemerintah menyampaikan terdapat 20.867 kasus...,negatif,KOMPAS_9945
4,2022-01-25,update 25 januari 7.483 kasus suspek covid-19 ...,"pemerintah mencatat, hingga selasa 25 1 2022 ...",negatif,KOMPAS_9936


,date,title,content,label,article_id
795,2024-09-23,jokowi ingin dunia bergantung ke indonesia unt...,presiden joko widodo berambisi menjadikan ind...,positif,KOMPAS_2878
796,2024-09-26,"ksau untuk jaga wilayah udara indonesia, tni h...","jakarta, kompas.com -kepala staf tni angkatan ...",netral,KOMPAS_2852
797,2024-09-27,"kembangkan pesawat n219 amfibi, pt di kami sad...",direktur produksi pt dirgantara indonesia pt ...,positif,KOMPAS_2842
798,2024-09-29,"ke ntb, jokowi bakal saksikan motogp pertamina...",presiden ri joko widodo bertolak ke provinsi ...,netral,KOMPAS_2833
799,2024-09-29,populer nasional indonesia walk out saat netan...,pemerintah indonesia konsisten dalam mendukun...,negatif,KOMPAS_2835


In [8]:
#normalisasi data, label jadiin 1 0 -1, title+content jadi ke text


# gabungin title content buat input ke IndoBERT
df["text"] = (
    df["title"].str.strip() + ". " +
    df["content"].str.strip()
).str.strip()

#normalized label
sentiment_map = {
    "positif": 1,
    "netral": 0,
    "negatif": -1
}

df["label"] = df["label"].map(sentiment_map)


In [9]:
df.head()

,date,title,content,label,article_id,text
0,2022-01-23,polemik arteria dahlan jadi pembelajaran kader...,sekretaris jenderal pdi-p hasto kristiyanto m...,0,KOMPAS_9963,polemik arteria dahlan jadi pembelajaran kader...
1,2022-01-23,"hut ke-75 megawati, pramono anung politik ibu ...",mantan sekretaris jenderal pdi perjuangan pra...,0,KOMPAS_9962,"hut ke-75 megawati, pramono anung politik ibu ..."
2,2022-01-24,pengamat tebak pesan anies ke giring kalau sum...,direktur eksekutif parameter politik indonesi...,0,KOMPAS_9956,pengamat tebak pesan anies ke giring kalau sum...
3,2022-01-24,update 20.867 kasus aktif covid-19 di indonesia,pemerintah menyampaikan terdapat 20.867 kasus...,-1,KOMPAS_9945,update 20.867 kasus aktif covid-19 di indonesi...
4,2022-01-25,update 25 januari 7.483 kasus suspek covid-19 ...,"pemerintah mencatat, hingga selasa 25 1 2022 ...",-1,KOMPAS_9936,update 25 januari 7.483 kasus suspek covid-19 ...


In [10]:
# jadiin csv
df.to_csv(
    "kompas_cleaned_labeled_manual.csv",
    index=False,
    encoding="utf-8"
)
